# LFT / QuantFolio — Analyse complète de portefeuille (édition Jupyter)

Ce notebook reproduit, étape par étape, le pipeline quantitatif du projet **LFT** :

1. **Données** — prix réels via yfinance + cache SQLite (repli synthétique hors-ligne)
2. **Métriques** — performance et risque par actif et pour votre portefeuille
3. **Health check** — concentration, diversification, contributions au risque
4. **CAPM & Fama-French** — alpha, beta, expositions factorielles
5. **Optimisation Markowitz** — MaxSharpe, MinVol, RiskParity, frontière efficiente
6. **Backtest** — comparaison des stratégies (in-sample, pédagogique)
7. **Walk-forward** — le backtest honnête, sans look-ahead
8. **Monte Carlo** — projection réaliste à 3 ans (GBM + bootstrap)
9. **Régimes de marché** — classification causale + matrice de transition
10. **AI Analyst** — rapport rédigé, long et détaillé

> **Utilisation** : modifiez la cellule *Configuration* (vos tickers et quantités),
> puis *Run All*. Sans connexion internet, les données synthétiques prennent le relais
> (clairement signalé). Rien ici ne constitue un conseil en investissement.

## 0. Configuration — adaptez cette cellule à votre portefeuille

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Votre portefeuille : ticker -> nombre de titres (modifiez-moi !) ---
POSITIONS = {
    "AAPL": 25, "MSFT": 15, "AMZN": 20, "JNJ": 30, "TLT": 40, "GLD": 18,
}
CASH = 4_500.0            # cash non investi (informatif)
BENCHMARK = "SPY"         # indice de référence
START = "2020-01-01"
END = pd.Timestamp.today().date().isoformat()   # aujourd'hui
RISK_FREE = 0.03          # taux sans risque annuel
TC_BPS = 10.0             # coûts de transaction (points de base)
MAX_WEIGHT = 0.25         # poids max par actif pour les stratégies optimisées
INITIAL_VALUE = 10_000    # valeur initiale des backtests / projections

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 160)
print(f"Univers : {sorted(POSITIONS)} + benchmark {BENCHMARK}")
print(f"Période : {START} -> {END} | rf {RISK_FREE:.1%} | coûts {TC_BPS:.0f} bps")

## 1. Données de marché

Les prix sont chargés via le **store SQLite local** (`quantfolio_prices.db`) :
téléchargement incrémental via yfinance, cache réutilisé entre les sessions,
repli sur des données synthétiques si hors-ligne (jamais mises en cache).

In [ ]:
from quantfolio.store import PriceStore
from quantfolio import data

store = PriceStore(PROJECT_DIR / "quantfolio_prices.db")
tickers = sorted(set(POSITIONS) | {BENCHMARK})
prices, source = store.get_prices(tickers, START, END)
print(f"Source : {source} | {len(prices)} jours de bourse | "
      f"{prices.index[0].date()} -> {prices.index[-1].date()}")
if source == "synthetic":
    print("(!) Mode hors-ligne : données synthétiques, PAS de vrais prix.")
prices.tail()

### Poids actuels du portefeuille (aux derniers prix)

In [ ]:
last = prices.iloc[-1]
held = [t for t in POSITIONS if t in prices.columns]
values = pd.Series({t: POSITIONS[t] * float(last[t]) for t in held})
weights = values / values.sum()

returns = data.to_returns(prices)
asset_returns = returns[held]
bench_returns = returns[BENCHMARK]
port_returns = (asset_returns * weights).sum(axis=1).rename("MyPortfolio")

overview = pd.DataFrame({
    "shares": pd.Series(POSITIONS), "last price": last,
    "value": values, "weight": weights,
}).dropna()
print(f"Valeur investie : {values.sum():,.2f} | cash : {CASH:,.2f} | "
      f"total : {values.sum() + CASH:,.2f}")
overview

## 2. Métriques par actif — performance et risque

Toutes les métriques sont annualisées sur 252 jours de bourse.
Détails des formules : `docs/THEORY.md`.

In [ ]:
from quantfolio import metrics

stats = metrics.summary(asset_returns, bench_returns, RISK_FREE)
display(stats.round(3))

port_stats = metrics.summary(port_returns, bench_returns, RISK_FREE)
print("Votre portefeuille (poids actuels) :")
port_stats.round(3)

## 3. Health check — concentration, diversification, risque

Le diagnostic structurel du portefeuille : indice de Herfindahl (HHI),
nombre effectif de positions, ratio de diversification, corrélation moyenne,
et **contributions au risque** (qui porte vraiment le risque ?).

In [ ]:
from quantfolio import advisor, optimization as opt

health, flags = advisor.health_check(weights, asset_returns, bench_returns, RISK_FREE)
display(health.to_frame())
print("Alertes :")
for f in flags:
    print("  -", f)

mu_ann, cov_ann = opt.annualized_inputs(asset_returns, shrinkage=True,
                                        mean_shrinkage=0.5)
print("\nContributions au risque (poids vs part du risque) :")
advisor.risk_contributions(weights, cov_ann).round(4)

## 4. CAPM & Fama-French

Régressions OLS sur rendements excédentaires journaliers. L'alpha annualisé
mesure la performance au-delà de l'exposition au marché ; les chargements
Fama-French révèlent les biais style (small-cap, value...).

In [ ]:
from quantfolio import capm, factors

capm_res = capm.capm_table(asset_returns, bench_returns, RISK_FREE)
display(capm_res.round(3))

ff, ff_source = factors.get_ff_factors(START, END, model="3", index=returns.index)
ff = ff.reindex(returns.index).dropna()
if ff_source == "synthetic":
    ff["Mkt-RF"] = bench_returns.reindex(ff.index) - ff["RF"]
print(f"(facteurs : {ff_source})")
ff_res = factors.ff_table(asset_returns.loc[ff.index], ff)
ff_res.round(3)

## 5. Optimisation Markowitz

Stratégies calculées sur covariance **Ledoit-Wolf** (plus robuste) avec plafond
de poids par actif. Le portefeuille *Current* est le vôtre.

In [ ]:
n = len(mu_ann)
bounds = opt.weight_bounds(n, max(MAX_WEIGHT, 1 / n + 0.01))
strategies = {
    "MaxSharpe": opt.max_sharpe_weights(mu_ann, cov_ann, RISK_FREE, bounds),
    "MinVol": opt.min_volatility_weights(cov_ann, bounds),
    "RiskParity": opt.risk_parity_weights(cov_ann, bounds),
    "EqualWeight": opt.equal_weights(held),
    "Current": weights,
}
alloc = pd.DataFrame(strategies)
display(alloc.round(3))

rows = {}
for name, w in strategies.items():
    wv = w.reindex(mu_ann.index).fillna(0.0).values
    r_, v_ = opt.portfolio_return(wv, mu_ann), opt.portfolio_volatility(wv, cov_ann)
    rows[name] = {"return": r_, "volatility": v_, "Sharpe": (r_ - RISK_FREE) / v_}
pd.DataFrame(rows).T.round(3)

### Frontière efficiente + matrice de corrélation

In [ ]:
from quantfolio import report
from IPython.display import Image, display

frontier = opt.efficient_frontier(mu_ann, cov_ann, n_points=60, bounds=bounds)
cloud = opt.random_portfolios(mu_ann, cov_ann, n=4000, bounds=bounds)
highlights = {
    k: (opt.portfolio_volatility(strategies[k].reindex(mu_ann.index).fillna(0).values, cov_ann),
        opt.portfolio_return(strategies[k].reindex(mu_ann.index).fillna(0).values, mu_ann))
    for k in ("MaxSharpe", "MinVol")
}
report.plot_efficient_frontier(frontier, cloud, highlights, None,
                               OUTPUT_DIR / "nb_frontier.png")
report.plot_correlation_matrix(asset_returns, OUTPUT_DIR / "nb_correlation.png")
display(Image(filename=str(OUTPUT_DIR / "nb_frontier.png")))
display(Image(filename=str(OUTPUT_DIR / "nb_correlation.png")))

## 6. Backtest comparatif (in-sample — pédagogique)

Rebalancement mensuel, coûts de 10 bps. **Attention** : les poids optimisés
utilisent toute la période (look-ahead assumé, à but pédagogique) — la section
suivante montre la version honnête.

In [ ]:
from quantfolio import backtest

bt_strategies = {k: v for k, v in strategies.items() if k != "Current"}
curves = backtest.compare_strategies(prices[held], bt_strategies,
                                     INITIAL_VALUE, rebalance="M", tc_bps=TC_BPS)
curves[BENCHMARK] = (INITIAL_VALUE * (1 + bench_returns).cumprod()).reindex(curves.index)

bt_returns = curves.pct_change().dropna()
bt_stats = metrics.summary(bt_returns, bench_returns, RISK_FREE)
display(bt_stats.loc[["CAGR", "Annualized volatility", "Sharpe",
                      "Max drawdown", "Calmar"]].round(3))

report.plot_equity_curves(curves, OUTPUT_DIR / "nb_equity.png")
report.plot_drawdowns({c: bt_returns[c] for c in curves.columns},
                      OUTPUT_DIR / "nb_drawdowns.png")
display(Image(filename=str(OUTPUT_DIR / "nb_equity.png")))
display(Image(filename=str(OUTPUT_DIR / "nb_drawdowns.png")))

## 7. Walk-forward — le backtest honnête (hors-échantillon)

À chaque rebalancement, les poids ne sont calculés que sur une fenêtre roulante
de 252 jours **passés**. L'écart avec la section 6 mesure l'erreur d'estimation —
c'est la leçon la plus importante de la finance quantitative.

In [ ]:
from quantfolio import walkforward as wf

wf_strategies = {
    "MaxSharpe WF": wf.make_max_sharpe(RISK_FREE, max(MAX_WEIGHT, 1 / n + 0.01)),
    "MinVol WF": wf.make_min_vol(max(MAX_WEIGHT, 1 / n + 0.01)),
    "EqualWeight": wf.make_equal_weight(),
}
wf_curves, wf_results = wf.compare_walk_forward(
    prices[held], wf_strategies, lookback=252, tc_bps=TC_BPS)
wf_curves[BENCHMARK] = INITIAL_VALUE * (1 + bench_returns.reindex(wf_curves.index)).cumprod()

wf_stats = metrics.summary(wf_curves.pct_change().dropna(),
                           bench_returns.reindex(wf_curves.index), RISK_FREE)
display(wf_stats.loc[["CAGR", "Annualized volatility", "Sharpe",
                      "Max drawdown", "Calmar"]].round(3))
report.plot_equity_curves(wf_curves, OUTPUT_DIR / "nb_walkforward.png",
                          title="Walk-forward (out-of-sample)")
display(Image(filename=str(OUTPUT_DIR / "nb_walkforward.png")))

## 8. Projection Monte Carlo — 3 ans, 2 × 20 000 scénarios

Deux moteurs complémentaires, tous deux **réalistes** : rendements espérés
ancrés (CAPM + plafond), nets de frais, en nominal et en réel (inflation).
Le GBM agrège la covariance complète ; le bootstrap par blocs préserve les
queues épaisses et le clustering de volatilité.

In [ ]:
from quantfolio import montecarlo as mc

horizon = 3 * 252
mc_gbm = mc.simulate_gbm(asset_returns, weights, INITIAL_VALUE, horizon,
                         n_sims=20000, risk_free=RISK_FREE)
mc_boot = mc.simulate_bootstrap(asset_returns, weights, INITIAL_VALUE, horizon,
                                n_sims=20000, risk_free=RISK_FREE)
mc_summary = pd.concat([mc_gbm.summary(), mc_boot.summary()], axis=1)
mc_summary.columns = ["GBM", "Bootstrap"]
display(mc_summary)

report.plot_monte_carlo(mc_gbm, OUTPUT_DIR / "nb_mc_gbm.png")
report.plot_monte_carlo(mc_boot, OUTPUT_DIR / "nb_mc_boot.png")
display(Image(filename=str(OUTPUT_DIR / "nb_mc_gbm.png")))
display(Image(filename=str(OUTPUT_DIR / "nb_mc_boot.png")))

## 9. Régimes de marché

Classification **causale** de chaque jour en 4 régimes (tendance × volatilité,
seuil = médiane expansive, donc sans look-ahead), comportement de votre
portefeuille par régime, et matrice de transition markovienne.

In [ ]:
from quantfolio import regime

regimes = regime.classify_regimes(bench_returns)
cur_regime, run_days = regime.current_regime(regimes)
print(f"Régime actuel : {cur_regime} (depuis {run_days} jours de bourse)")

rs = regime.regime_stats(port_returns, regimes)
display(rs)

trans = regime.transition_matrix(regimes)
print("Matrice de transition journalière P[demain | aujourd'hui] :")
display(trans.round(3))
print(f"Durée attendue du régime '{cur_regime}' : "
      f"{regime.expected_regime_duration(trans, cur_regime):.0f} jours")

## 10. AI Analyst — le rapport complet

Le moteur rédige une **analyse longue et structurée** : synthèse, performance,
risque, diversification, scénarios, forces, faiblesses, plan d'action,
checklist de suivi et glossaire.
Avec `ANTHROPIC_API_KEY` définie, c'est le modèle Claude qui rédige (encore
plus riche) ; sinon, le moteur de règles hors-ligne prend le relais.

In [ ]:
from quantfolio.ai_analyst import AIAnalyst

analyst = AIAnalyst()
print(f"Mode : {'Claude API' if analyst.api_key else 'moteur hors-ligne'}\n")

w_disp = {k: round(float(v), 3) for k, v in weights.items()}
report_text = analyst.explain(
    port_stats, health, mc_summary,
    context=(f"Portefeuille : {w_disp} | cash {CASH:,.0f} | "
             f"alertes : {'; '.join(flags)} | REGIME : {cur_regime} depuis "
             f"{run_days} jours | données {source} {START} -> {END}"),
)
print(report_text)

out_path = OUTPUT_DIR / "ai_analysis.txt"
out_path.write_text(report_text, encoding="utf-8")
print(f"\nRapport sauvegardé : {out_path}")

## Pour aller plus loin

- **Votre vrai portefeuille** : remplacez `POSITIONS` dans la cellule 0, ou
  importez depuis l'app (`app.py` / `gui.py` — sync IBKR en lecture seule).
- **Reproductibilité** : les données réelles sont en cache dans
  `quantfolio_prices.db` ; supprimez le fichier pour repartir de zéro.
- **Théorie** : `docs/THEORY.md` détaille toutes les formules utilisées.
- **Tests** : le dossier `tests/` contient 121 vérifications exécutables.

*Projet éducatif — rien de tout ceci n'est un conseil en investissement.*